In [1]:
%pip install llama-index-core llama-index-llms-cohere llama-index-readers-file llama-index-embeddings-cohere llama-index-vector-stores-pinecone gradio

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Environment and Settings

import os
import ssl
import httpx
import json
import urllib3
from dotenv import load_dotenv

from llama_index.core import SimpleDirectoryReader, Settings
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.ingestion import IngestionPipeline
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.llms.cohere import Cohere
from llama_index.core import VectorStoreIndex, PromptTemplate
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Event
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.llms import ChatMessage

from pinecone import Pinecone, ServerlessSpec
import gradio as gr
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime, timedelta

COHERE_MODEL_NAME = "embed-multilingual-v3.0"
PINECONE_INDEX_NAME = "agentic-coding-index"
EMBEDDING_DIMENSION = 1024
TARGET_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'meta-observer-target'))

TOOLS_CONFIG = {
    "Cursor": ".cursor",
    "Claude Code": ".claude"
}

load_dotenv()

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

import httpx

# 1. מעקף עבור הלקוח הסינכרוני (Client)
if not hasattr(httpx.Client, "_is_patched"):
    original_sync_init = httpx.Client.__init__
    def patched_sync_init(self, *args, **kwargs):
        kwargs['verify'] = False
        kwargs['timeout'] = 120.0  # זמן המתנה ארוך של 2 דקות
        original_sync_init(self, *args, **kwargs)
    httpx.Client.__init__ = patched_sync_init
    httpx.Client._is_patched = True
    
# 2. מעקף עבור הלקוח האסינכרוני (AsyncClient) עם הגדרות רשת יציבות
if not hasattr(httpx.AsyncClient, "_is_patched"):
    original_async_init = httpx.AsyncClient.__init__
    def patched_async_init(self, *args, **kwargs):
        kwargs['verify'] = False
        kwargs['timeout'] = httpx.Timeout(180.0, connect=60.0) # זמן המתנה ארוך יותר
        # הוספת מגבלות כדי למנוע ניתוקים של Windows/NetFree
        kwargs['limits'] = httpx.Limits(max_keepalive_connections=5, max_connections=10, keepalive_expiry=30.0)
        original_async_init(self, *args, **kwargs)
    httpx.AsyncClient.__init__ = patched_async_init
    httpx.AsyncClient._is_patched = True


Settings.embed_model = CohereEmbedding(
    model_name="embed-multilingual-v3.0",
    api_key=os.getenv("COHERE_API_KEY"),
    timeout=120.0
)

print("Environment and Settings initialized.")

c:\Users\user1\Desktop\תכנות\תשפו_בסד\AI\rag\ragProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment and Settings initialized.


In [3]:
#Pinecone

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"),
    ssl_verify=False
)

if PINECONE_INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    print(f"Creating new Pinecone index: {PINECONE_INDEX_NAME}...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1") 
    )

pinecone_index = pc.Index(PINECONE_INDEX_NAME)
print("Successfully connected to Pinecone.")


Successfully connected to Pinecone.


In [4]:
# Data Loading & Add Metadata

all_documents = []
    
for tool_name, folder_name in TOOLS_CONFIG.items():
    folder_path = os.path.join(TARGET_DIR, folder_name)
    
    if os.path.exists(folder_path):

        reader = SimpleDirectoryReader(
            input_dir=folder_path, 
            required_exts=[".md"], 
            recursive=True,
            exclude_hidden=False
        )
        docs = reader.load_data()

        for doc in docs:
            doc.metadata["tool_name"] = tool_name
            
        all_documents.extend(docs)
        print(f"Loaded {len(docs)} documents from {tool_name}.")
    else:
        print(f"Folder not found for {tool_name}: {folder_path}")

Loaded 1 documents from Cursor.
Loaded 1 documents from Claude Code.


In [5]:
# Ingestion Pipeline

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
pipeline = IngestionPipeline(
    transformations=[
        MarkdownNodeParser(),
        Settings.embed_model,
    ],
    vector_store=vector_store
)

nodes = pipeline.run(documents=all_documents)
print(f"Success! Processed, embedded, and indexed {len(nodes)} nodes (chunks) into Pinecone.")

2026-05-04 15:10:20,621 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-05-04 15:10:21,184 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 11/11 [00:08<00:00,  1.27it/s]

Success! Processed, embedded, and indexed 11 nodes (chunks) into Pinecone.


In [6]:
# LLM and Index Initialization

Settings.llm = Cohere(
    model="command-r-08-2024",  
    api_key=os.getenv("COHERE_API_KEY"),
    timeout=120.0,
    max_tokens=512,
    system_prompt="You are an expert AI assistant for developers.\n" \
                  "The context documents are usually in English, but the user will ask questions in Hebrew.\n" \
                  "You must search the English context, find the relevant information, and answer the user in fluent Hebrew."
)
Settings.context_window = 8192  
Settings.num_output = 512

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
print("LLM and Index are ready for querying.")


LLM and Index are ready for querying.


In [7]:
# Workflow Definition

class InputValidatedEvent(Event):
    query: str
    chat_history: list

class RetrieveEvent(Event):
    query: str
    nodes: list

class FallbackSearchEvent(Event):
    query: str
    chat_history: list

class SemanticSearchEvent(Event):
    query: str
    chat_history: list

class StructuredSearchEvent(Event):
    query: str
    chat_history: list

class StructuredSearchParams(BaseModel):
    entity_type: str = Field(description="Must be exactly one of: 'decisions', 'rules', 'warnings'")
    keyword: str = Field(default="", description="A word to filter by. If none, return an empty string '', NEVER null.")
    days_back: int = Field(default=0, description="Number of days to look back. If none, return 0, NEVER null.")

In [8]:
# import json
# import os
# from datetime import datetime, timedelta

# def execute_structured_search(entity_type: str, keyword: str = "", days_back: int = 0) -> str:
#     try:
#         file_path = os.path.abspath("extracted_knowledge.json")
#         print(f"DEBUG - Looking for file at: {file_path}")
        
#         if not os.path.exists(file_path):
#              return f"שגיאה: קובץ הנתונים המובנים לא נמצא בנתיב: {file_path}"
             
#         with open(file_path, "r", encoding="utf-8") as f:
#             data = json.load(f)
            
#         items = data.get("items", {}).get(entity_type, [])
#         print(f"DEBUG - Found {len(items)} items of type {entity_type} in file.")
        
#         if keyword:
#             items = [item for item in items if keyword.lower() in str(item).lower()]
            
#         if days_back > 0:
#             cutoff_date = datetime.now() - timedelta(days=days_back)
#             filtered_items = []
#             for item in items:
#                 observed_str = item.get("observed_at", "")
#                 if observed_str:
#                     try:
#                         item_date = datetime.fromisoformat(observed_str)
#                         if item_date >= cutoff_date:
#                             filtered_items.append(item)
#                     except ValueError as ve:
#                         print(f"DEBUG - Date parsing error: {ve}")
#                         continue
#             items = filtered_items
            
#         if not items:
#             return f"לא נמצאו נתונים מסוג {entity_type} התואמים לבקשתך."
            
#         return json.dumps(items, ensure_ascii=False, indent=2)
        
#     except Exception as e:
#         error_msg = f"שגיאה בשליפת הנתונים המובנים: {str(e)}"
#         print(f"DEBUG - Exception occurred: {error_msg}")
#         return error_msg

# print("Structured Search Function is ready (With Exception Debugging).")

import json
import os

def execute_structured_search(entity_type: str, keyword: str = "", days_back: int = 0) -> str:
    try:
        file_path = os.path.abspath("extracted_knowledge.json")
        print(f"DEBUG - Reading from: {file_path}")
        
        if not os.path.exists(file_path):
             return f"שגיאה: קובץ לא נמצא."
             
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        items = data.get("items", {}).get(entity_type, [])
        print(f"DEBUG - Items before return: {len(items)}")
        
        if not items:
            return f"לא נמצאו נתונים מסוג {entity_type}."
            
        # מחזירים את הנתונים ישר, בלי שום סינון!
        return json.dumps(items, ensure_ascii=False, indent=2)
        
    except Exception as e:
        return f"שגיאה: {str(e)}"

print("Structured Search Function is ready (No filters).")



Structured Search Function is ready (No filters).


In [9]:
# RAG Workflow Definition

class RAGWorkflow(Workflow):
    
    def __init__(self, rag_index, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.rag_index = rag_index

    @step
    async def validate_input(self, ev: StartEvent) -> InputValidatedEvent | StopEvent:
        query = ev.get("query")
        chat_history = ev.get("chat_history", [])
        
        if not query or len(query.strip()) < 3:
            return StopEvent(result="❌ שגיאה: השאלה קצרה מדי. אנא נסחי שאלה ברורה.")

        # --- טיפול בהיסטוריית שיחה (Query Rewriting) ---
        if chat_history:
            print("📝 משכתב את השאלה בהתבסס על היסטוריית השיחה...")
            recent_history = chat_history[-3:]
            
            # בניית היסטוריית השיחה בצורה בטוחה שתומכת בכל הגרסאות של Gradio
            history_lines = []
            for item in recent_history:
                if isinstance(item, (list, tuple)) and len(item) >= 2:
                    history_lines.append(f"שאלה: {item[0]}\nתשובה: {item[1]}")
                elif isinstance(item, dict):
                    history_lines.append(f"{item.get('role', 'msg')}: {item.get('content', '')}")
                else:
                    try: # תמיכה באובייקטים של Gradio 4+
                        history_lines.append(f"{item.role}: {item.content}")
                    except:
                        history_lines.append(str(item))
            
            history_str = "\n".join(history_lines)
            
            rewrite_prompt = (
                "נסח מחדש את השאלה הנוכחית כך שתהיה ברורה ועצמאית לחלוטין, "
                "תוך שימוש בהקשר מתוך היסטוריית השיחה.\n"
                f"היסטוריה:\n{history_str}\n\n"
                f"שאלה נוכחית: {query}\n"
                "שאלה עצמאית מנוסחת מחדש:"
            )
            
            response = await Settings.llm.achat([ChatMessage(role="user", content=rewrite_prompt)])
            query = str(response.message.content).strip()
            print(f"✨ השאלה שוכתבה ל: {query}")
           
        return InputValidatedEvent(query=query, chat_history=chat_history)



    @step
    async def route_query(self, ev: InputValidatedEvent) -> SemanticSearchEvent | StructuredSearchEvent:
        prompt = (
            "Classify the following user query into exactly one of these two categories:\n"
            "1. 'structured' - if the user asks for a list of items (decisions, rules, warnings), recent updates, or filtering.\n"
            "2. 'semantic' - if the user asks conceptual questions (how to, why, explanations).\n"
            "Respond with ONLY ONE WORD: 'structured' or 'semantic'.\n"
            f"Query: {ev.query}"
        )
        
        # שימוש ב-LLM כדי להחליט על נתיב
        response = await Settings.llm.achat([ChatMessage(role="user", content=prompt)])
        decision = str(response.message.content).strip().lower()
        
        print(f"🔀 החלטת ניתוב: {decision}")
        
        if "structured" in decision:
            return StructuredSearchEvent(query=ev.query, chat_history=ev.chat_history)
        else:
            return SemanticSearchEvent(query=ev.query, chat_history=ev.chat_history)
    
    @step
    async def retrieve_data(self, ev: SemanticSearchEvent) -> RetrieveEvent | FallbackSearchEvent | StopEvent:
        retriever = self.rag_index.as_retriever(similarity_top_k=6)
        nodes = await retriever.aretrieve(ev.query)
        
        if not nodes:
            return StopEvent(result="לא מצאתי מידע רלוונטי במסמכי התיעוד לשאלה זו.")
            
        processor = SimilarityPostprocessor(similarity_cutoff=0.5)
        high_quality_nodes = processor.postprocess_nodes(nodes)
        
        if not high_quality_nodes:
            return FallbackSearchEvent(query=ev.query, chat_history=ev.chat_history)
            
        return RetrieveEvent(query=ev.query, nodes=high_quality_nodes)

    @step
    async def fallback_search(self, ev: FallbackSearchEvent) -> RetrieveEvent | StopEvent:
        print("🔄 מפעיל חיפוש מורחב (Fallback)...")
        retriever = self.rag_index.as_retriever(similarity_top_k=10)
        nodes = await retriever.aretrieve(ev.query)
        
        if not nodes:
            return StopEvent(result="לא מצאתי מידע גם לאחר חיפוש מורחב. ייתכן שנדרש הקשר נוסף לשאלה.")
            
        return RetrieveEvent(query=ev.query, nodes=nodes)

    @step
    async def synthesize_response(self, ev: RetrieveEvent) -> StopEvent:
        print("✍️ מנסח תשובה סופית (Synthesizing)...")
        # 1. חיבור הטקסט מכל המסמכים שמצאנו
        docs_text = "\n\n".join([node.text for node in ev.nodes])
        
        # 2. נגדיר את הפרומפט (נמלא אותו אחר כך)
        prompt_text = f"ענה על השאלה: {ev.query} בהתבסס אך ורק על המידע הבא:\n{docs_text}"
        
        try:
            # 3. שליחה ישירה ומהירה למודל
            res = await Settings.llm.achat([ChatMessage(role="user", content=prompt_text)])
            ans = str(res.message.content)
            
            # 4. הוספת מקורות
            srcs = list(set([n.metadata.get('tool_name', 'Unknown') for n in ev.nodes]))
            return StopEvent(result=ans + f"\n\n*מקורות: {', '.join(srcs)}*")
            
        except Exception as err:
            return StopEvent(result=f"❌ שגיאה בניסוח: {str(err)}")
        
    # @step
    # async def structured_search_step(self, ev: StructuredSearchEvent) -> StopEvent:
    #     print("📊 מפעיל חיפוש מובנה (Structured)...")
    #     P1 = f"Based on the user query, extract the search parameters in a JSON format with the following keys: 'entity_type' (must be one of 'decisions', 'rules', 'warnings'), 'keyword' (a string to filter by, can be empty), and 'days_back' (an integer for how many days back to look, can be 0). User query: {ev.query}\nRespond ONLY with the JSON object, no explanations."
    #     try:
    #         # 1. שליחת בקשה למודל
    #         res = await Settings.llm.achat([ChatMessage(role="user", content=P1)])
    #         txt = str(res.message.content)
            
    #         # 2. חילוץ בטוח של JSON
    #         s = txt.find("{")
    #         e = txt.rfind("}") + 1
            
    #         if s == -1 or e == 0:
    #             return StopEvent(result="❌ שגיאה: המודל לא החזיר פרמטרי חיפוש בפורמט תקין.")
                
    #         data = json.loads(txt[s:e])
            
    #         # 3. ביצוע החיפוש
    #         raw = execute_structured_search(
    #             entity_type=data.get("entity_type", "decisions"),
    #             keyword=data.get("keyword", ""),
    #             days_back=data.get("days_back", 0)
    #         )
    #         P2 = f"Answer in Hebrew based ONLY on the Data. If the Data says no results were found, tell the user politely and DO NOT invent information.\nData: {raw}\nQuery: {ev.query}"
    #         # 4. ניסוח תשובה סופית
    #         final_res = await Settings.llm.achat([ChatMessage(role="user", content=P2)])
    #         return StopEvent(result=str(final_res.message.content) + "\n\n*מקור: מאגר מובנה*")
            
    #     except json.JSONDecodeError:
    #         return StopEvent(result="❌ שגיאה: כשל בפענוח הנתונים מהמודל.")
    #     except Exception as err:
    #         return StopEvent(result=f"❌ שגיאה כללית בחיפוש המובנה: {str(err)}")

    @step
    async def structured_search_step(self, ev: StructuredSearchEvent) -> StopEvent:
        print("📊 מפעיל חיפוש מובנה...")
        import json
        
        P1 = f"Based on the user query, extract the search parameters in a JSON format with the following keys: 'entity_type' (must be exactly one of 'decisions', 'rules', 'warnings'), 'keyword' (a string to filter by, can be empty), and 'days_back' (an integer for how many days back to look, can be 0). User query: {ev.query}\nRespond ONLY with the JSON object, no explanations."
        
        try:
            res = await Settings.llm.achat([ChatMessage(role="user", content=P1)])
            txt = str(res.message.content)
            
            s = txt.find("{")
            e = txt.rfind("}") + 1
            
            if s == -1 or e == 0:
                return StopEvent(result="❌ שגיאה: המודל לא החזיר פרמטרי חיפוש בפורמט תקין.")
                
            data = json.loads(txt[s:e])
            
            # הדפסה לבדיקה: מה המודל ביקש?
            print(f"DEBUG - Extracted Params: {data}")
            
            # תיקון אוטומטי של טעויות נפוצות
            entity = data.get("entity_type", "decisions").lower()
            if "rule" in entity: entity = "rules"
            elif "warn" in entity: entity = "warnings"
            else: entity = "decisions"

            raw = execute_structured_search(
                entity_type=data.get("entity_type", "decisions"),
                keyword=data.get("keyword", ""),
                days_back=data.get("days_back", 0)
            )
            
            P2 = f"""
            You are a strict data reporter. 
            I am giving you raw JSON data that contains the EXACT decisions the user asked for.
            You MUST output this data to the user in fluent Hebrew.
            DO NOT analyze if the data is a "decision" or not. If it is in the data, it IS a decision.
            List every single item from the data below.
            
            Data:
            {raw}
            
            User Query: {ev.query}
            """

            final_res = await Settings.llm.achat([ChatMessage(role="user", content=P2)])
            return StopEvent(result=str(final_res.message.content) + "\n\n*מקור: מאגר מובנה*")
            
        except json.JSONDecodeError:
            return StopEvent(result="❌ שגיאה: כשל בפענוח הנתונים מהמודל.")
        except Exception as err:
            return StopEvent(result=f"❌ שגיאה כללית בחיפוש המובנה: {str(err)}")


workflow = RAGWorkflow(rag_index=index, timeout=180.0)


In [10]:
# PydanticSchemas for Data Extraction

class SourceInfo(BaseModel):
    tool: str = Field(description="שם כלי הקידוד, למשל: cursor, claude_code")
    file: str= Field(description="הנתיב לקובץ ממנו חולץ המידע")

class Decision(BaseModel):
    id: str = Field(description="מזהה ייחודי להחלטה, למשל: dec-001")
    title: str = Field(description="כותרת קצרה של ההחלטה הטכנית")
    summary: str = Field(description="סיכום תמציתי של ההחלטה שהתקבלה")
    tags: List[str] = Field(description="רשימת תגיות, למשל: db, architecture")
    source: SourceInfo
    observed_at:str = Field(description="תאריך ושעה בפורמט ISO 8601")

class Rule(BaseModel):
    id: str = Field(description="מזהה ייחודי לכלל, למשל: rule-001")
    rule: str = Field(description="הגדרת הכלל או ההנחיה")
    scope: str = Field(description="התחום עליו חל הכלל, למשל: ui, backend")
    source: SourceInfo
    observed_at: str = Field(description="תאריך ושעה בפורמט ISO 8601")

class WarningItem(BaseModel):
   id: str = Field(description="מזהה ייחודי לאזהרה, למשל: warn-001")
   area: str = Field(description="האזור או הרכיב הרגיש במערכת")
   message: str = Field(description="תוכן האזהרה אותיאור הרגישות")
   severity: str = Field(description="רמת חומרה, למשל: low, medium, high")
   source: SourceInfo
   observed_at: str = Field(description="תאריך ושעה בפורמט ISO 8601")

class ExtractedItems(BaseModel):
    decisions: List[Decision] = Field(default_factory=list, description="החלטות טכניות")
    rules: List[Rule] = Field(default_factory=list, description="כללים והנחיות")
    warnings: List[WarningItem] = Field(default_factory=list, description="אזהרות ורגישויות")
   
print("Schemas defined successfully.")

Schemas defined successfully.


In [11]:

import time
from datetime import datetime
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core import Settings

# יצירת תוכנית חילוץ מבוססת LLM ו-Pydantic
extraction_program = LLMTextCompletionProgram.from_defaults(
    output_cls=ExtractedItems,
    llm=Settings.llm,
    prompt_template_str=(
        "Please extract the following information from the text.\n"
        "If a specific type of information is not present, leave its list empty.\n"
        "Text: {text}"
    ),
)

def run_extraction(nodes):
    print(f"Starting extraction on {len(nodes)} nodes...")
    all_extracted = {"items": {"decisions": [], "rules": [], "warnings": []}}
    
    for i, node in enumerate(nodes):
        max_retries = 3
        for attempt in range(max_retries):
            try:
                # חילוץ הנתונים מה-LLM
                extracted_data = extraction_program(text=node.text)
                
                # העשרת המקור (Source) אוטומטית מתוך המטא-דאטה של ה-Node
                tool_name = node.metadata.get("tool_name", "unknown")
                file_path = node.metadata.get("file_path", "unknown")
                current_time = datetime.now().isoformat()
                
                # הוספת הנתונים שחולצו למאגר הכללי (שימוש ב-model_dump במקום dict)
                for decision in extracted_data.decisions:
                    decision.source = SourceInfo(tool=tool_name, file=file_path)
                    decision.observed_at = current_time
                    all_extracted["items"]["decisions"].append(decision.model_dump())
                    
                for rule in extracted_data.rules:
                    rule.source = SourceInfo(tool=tool_name, file=file_path)
                    rule.observed_at = current_time
                    all_extracted["items"]["rules"].append(rule.model_dump())
                    
                for warning in extracted_data.warnings:
                    warning.source = SourceInfo(tool=tool_name, file=file_path)
                    warning.observed_at = current_time
                    all_extracted["items"]["warnings"].append(warning.model_dump())
                    
                print(f"Processed node {i+1}/{len(nodes)} successfully.")
                break  # יציאה מלולאת הניסיונות החוזרים אם הצלחנו
                
            except Exception as e:
                print(f"Attempt {attempt + 1} failed for node {i+1}: {e}")
                if attempt < max_retries - 1:
                    print("Waiting 5 seconds before retrying...")
                    time.sleep(5)
                else:
                    print(f"Skipping node {i+1} after {max_retries} failed attempts.")
            
    # שמירה לקובץ
    with open("extracted_knowledge.json", "w", encoding="utf-8") as f:
        json.dump(all_extracted, f, ensure_ascii=False, indent=2)
        
    print("Extraction complete. Data saved to extracted_knowledge.json")

# הפעלת החילוץ
# run_extraction(all_documents)
print("Data extraction process finished.")

Data extraction process finished.


In [12]:
# Gradio Chat Interface

async def chat_with_workflow(message, history):
    try:
        result = await workflow.run(query=message, chat_history=history)
        return str(result)
    except Exception as e:
        return f"❌ שגיאה בתשאול: {str(e)}"

print("🚀 Launching Gradio Chat Interface...")

demo = gr.ChatInterface(
    fn=chat_with_workflow,
    title="🤖 Agentic Docs RAG (Event-Driven)",
    description="שאלי אותי שאלות על החלטות הפיתוח, חוקי המערכת והארכיטקטורה מתוך קבצי התיעוד."
)

demo.launch()

🚀 Launching Gradio Chat Interface...


2026-05-04 15:10:30,374 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


2026-05-04 15:10:30,690 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


...

2026-05-04 15:34:50,739 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


🔄 מפעיל חיפוש מורחב (Fallback)...


2026-05-04 15:34:52,954 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


✍️ מנסח תשובה סופית (Synthesizing)...


2026-05-04 15:35:01,288 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
